In [24]:
from collections import deque
import numpy as np

In [25]:
#smooth buffer holds last 8 raw readings - FIFO queue
SMOOTH_SIZE = 8
smooth_buffer = deque([0.0] * SMOOTH_SIZE, maxlen=SMOOTH_SIZE)

In [26]:
smooth_buffer

deque([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], maxlen=8)

In [27]:
def midi_to_hz(note):
    return 440.0 * (2.0 ** ((note - 69) / 12.0))

In [28]:
midi_to_hz(69) #A4 note=69 440.0Hz

440.0

In [29]:
midi_to_hz(60)  #Middle C note=60 261.63Hz

261.6255653005986

In [30]:
midi_to_hz(57)  #A3 57 note=57 220Hz

220.0

In [74]:
duration=0.4
sample_rate=44100
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False) #17640 samples/time points to represent 0.4 seconds of audio
t

array([0.00000000e+00, 2.26757370e-05, 4.53514739e-05, ...,
       3.99931973e-01, 3.99954649e-01, 3.99977324e-01], shape=(17640,))

In [75]:
# 2* np.pi * freq * t
freq = midi_to_hz(69) #A4 note=69 440.0Hz

In [90]:
wave = 0.50 * np.sin(2 * np.pi * freq * t) # base note
wave

array([ 0.        ,  0.03132416,  0.06252526, ..., -0.09348072,
       -0.06252526, -0.03132416], shape=(17640,))

In [91]:
wave = 0.50 * np.sin(2 * np.pi * freq * t) # base note
wave += 0.25 * np.sin(2 * np.pi * freq * 2 * t) #1st overtone
wave += 0.12 * np.sin(2 * np.pi * freq * 3 * t)     # 2nd overtone
wave += 0.06 * np.sin(2 * np.pi * freq * 4 * t)     # 3rd overtone
wave

array([ 0.        ,  0.09991044,  0.19748455, ..., -0.29048035,
       -0.19748455, -0.09991044], shape=(17640,))

In [92]:
fade_curve = np.exp(-3.5 * t / duration)
wave = wave * fade_curve
wave

array([ 0.        ,  0.09989062,  0.1974062 , ..., -0.00877697,
       -0.00596588, -0.00301763], shape=(17640,))

In [93]:
wave = (wave * 32767).astype(np.int16)
wave

array([   0, 3273, 6468, ..., -287, -195,  -98],
      shape=(17640,), dtype=int16)

In [94]:
stereo = np.column_stack([wave, wave])
stereo

array([[   0,    0],
       [3273, 3273],
       [6468, 6468],
       ...,
       [-287, -287],
       [-195, -195],
       [ -98,  -98]], shape=(17640, 2), dtype=int16)

In [89]:
pygame.sndarray.make_sound(stereo).play()

In [39]:
import numpy as np
import pygame

In [40]:
pygame.mixer.init(frequency=44100, size=-16, channels=2, buffer=512)
pygame.mixer.set_num_channels(4)   # allow a few notes at once

In [41]:
def midi_to_hz(note):
    return 440.0 * (2.0 ** ((note - 69) / 12.0))

def make_piano_note(midi_note, duration=0.4, sample_rate=44100):
    freq = midi_to_hz(midi_note)
    t    = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

    # Combine harmonics to sound more like a piano
    wave  = 0.50 * np.sin(2 * np.pi * freq * t)         # base note
    wave += 0.25 * np.sin(2 * np.pi * freq * 2 * t)     # 1st overtone
    wave += 0.12 * np.sin(2 * np.pi * freq * 3 * t)     # 2nd overtone
    wave += 0.06 * np.sin(2 * np.pi * freq * 4 * t)     # 3rd overtone

    # Fade out so it sounds natural (not a hard cut)
    fade_curve = np.exp(-3.5 * t / duration)
    wave = wave * fade_curve

    # Scale to 16-bit audio range and make it stereo
    wave = (wave * 32767).astype(np.int16)
    stereo = np.column_stack([wave, wave])   # left + right channels
    return stereo

In [42]:
# C major pentatonic scale — spans 3 octaves, always sounds musical
# Every note here harmonises with every other — impossible to sound bad!
SCALE = [
    48, 50, 52, 55, 57,   # C3  D3  E3  G3  A3  (low — very relaxed)
    60, 62, 64, 67, 69,   # C4  D4  E4  G4  A4  (middle — mild tension)
    72, 74, 76, 79, 81,   # C5  D5  E5  G5  A5  (high — strong squeeze)
]
SCALE

[48, 50, 52, 55, 57, 60, 62, 64, 67, 69, 72, 74, 76, 79, 81]

In [43]:
len(SCALE)

15

In [44]:
note_sounds = {}
for note_num in SCALE:
    samples = make_piano_note(note_num, duration=0.5)
    sound   = pygame.sndarray.make_sound(samples)
    sound.set_volume(0.7)
    note_sounds[note_num] = sound
note_sounds

{48: <pygame.mixer.Sound at 0x120cadcb0>,
 50: <pygame.mixer.Sound at 0x105795e60>,
 52: <pygame.mixer.Sound at 0x121de6490>,
 55: <pygame.mixer.Sound at 0x121de61f0>,
 57: <pygame.mixer.Sound at 0x120006ee0>,
 60: <pygame.mixer.Sound at 0x121de71e0>,
 62: <pygame.mixer.Sound at 0x121de7540>,
 64: <pygame.mixer.Sound at 0x121de7450>,
 67: <pygame.mixer.Sound at 0x121de75a0>,
 69: <pygame.mixer.Sound at 0x121de6e20>,
 72: <pygame.mixer.Sound at 0x121de7030>,
 74: <pygame.mixer.Sound at 0x121de5a70>,
 76: <pygame.mixer.Sound at 0x121de7270>,
 79: <pygame.mixer.Sound at 0x121de73c0>,
 81: <pygame.mixer.Sound at 0x121de6040>}

In [45]:
note = 69
note_sounds[note].play()

In [46]:
note = 57
note_sounds[note].play()

In [13]:
note = 60
note_sounds[note].play()

In [14]:
note = 67
note_sounds[note].play()

In [15]:
import time
for note in note_sounds:
    note_sounds[note].play()
    time.sleep(1)

In [16]:
import time
for note in note_sounds:
    note_sounds[note].play()
    time.sleep(0.75)

In [17]:
import time
for note in note_sounds:
    note_sounds[note].play()
    time.sleep(0.5)

In [18]:
import time
for note in note_sounds:
    note_sounds[note].play()
    time.sleep(0.25)

In [19]:
import time
for note in note_sounds:
    note_sounds[note].play()
    time.sleep(0.125)

In [20]:
import time
for note in note_sounds:
    note_sounds[note].play()
    time.sleep(0.075)

In [21]:
import time
for note in note_sounds:
    note_sounds[note].play()
    time.sleep(0.015)